# TUE Reimbursement

This notebook demonstrates goal-oriented alignment visualization for the TUE Reimbursment case study.

- Models are loaded in the notebook.
- Handmade test cases are defined inline based on the traces discussed in the paper.



## Setup & Imports

In [114]:
import pandas as pd

import pm4py
from pm4py.objects.conversion.log import converter as log_converter
from pm4py.objects.log.obj import EventLog


from Semantics.goccva_pipeline import analyse

from Ui.goccva_ui import render_from_analysis, render_all_goal_oriented_alignments_from_analysis, render_case_distribution_matrix

from Semantics.goccva_helpers import sequences_to_event_log

from Semantics.istar_processor import read_istar_model
from Semantics.petri_net_processor import read_petri_net
from Semantics.event_mapping_from_csv import read_event_mapping_csv

from Semantics.goccva_filter import ComplianceStatus, TraceFilter

from pprint import pp

## Load Models

Load the goal model, process model, and mapping used by the case study.

In [115]:

# Paths used in the GoCCvA repository
goal_model_path = "content/TUEReimbursement/gm_huba_new_actor2.txt"
process_model_path = "content/TUEReimbursement/domestic_declaration_ilpn_updated.pnml"
mapping_path = "content/TUEReimbursement/mapping_huba_new_actor.csv"

goal_model = read_istar_model(str(goal_model_path), qualified=True)

petri_net = read_petri_net(str(process_model_path))

activity_mapping = read_event_mapping_csv(str(mapping_path))

# Fix minor inconsistencies in the mapping vs. the goal model (e.g., extra spaces, missing/extra parentheses, etc.)
# This should not be required as I have changed GMjcavi2.txt to make it compatiible with the mapping.
activity_mapping = goal_model.canonicalize_activity_mapping(activity_mapping) 


## Target Configuration

Define the target requirements to be evaluated. The make, break, and non-related sets are computed from the loaded goal model.

In [116]:
targets = [
    '(Admin) adequate declaration handling',
    '(Employee) Increase employee satisfaction',
]

## Activity Abbreviations

In [117]:
activity_abbreviations = {
    "Declaration REJECTED by ADMINISTRATION": "ra",
    "Payment Handled": "ph",
    "Declaration REJECTED by BUDGET OWNER": "rb",
    "Declaration SAVED by EMPLOYEE": "dsv",
    "Declaration APPROVED by ADMINISTRATION": "aa",
    "Declaration REJECTED by EMPLOYEE": "er",
    "Request Payment": "rp",
    "Declaration SUBMITTED by EMPLOYEE": "ds",
    "Declaration APPROVED by BUDGET OWNER": "ba",
    "Declaration FINAL_APPROVED by SUPERVISOR": "as",
    "Declaration REJECTED by SUPERVISOR": "rs",
    "Declaration APPROVED by PRE_APPROVER": "pa",
    "Declaration REJECTED by PRE_APPROVER": "rpa",
    "Declaration REJECTED by MISSING": "rm",
    "t_tau_rev": "tau",
}

print("Activity abbreviations configured")


Activity abbreviations configured


# Reading the .xes file


In [118]:
log_file_path = "content/TUEReimbursement/DomesticDeclarations.xes.gz"

full_log = log_converter.apply(pm4py.read_xes(str(log_file_path)), variant=log_converter.Variants.TO_EVENT_LOG)

print("Log file loaded")
print(len(full_log))

Log file loaded
10357


In [119]:
summary, detailed, contribution_to_targets = analyse(
    goal_model,
    petri_net,
    full_log,
    targets,
    activity_mapping,
    initial_marking=None,
 )

matrix_result = render_case_distribution_matrix(
    summary=summary,
    title="TUE Reimbursement Case Distribution Matrix",
    targets=targets,
)

print(matrix_result["counts"])

aligning log, completed variants ::   0%|          | 0/90 [00:00<?, ?it/s]

{'O+': 2411, 'O~': 0, 'O-': 185, 'N+': 6584, 'N~': 914, 'N-': 263}


## Compute the traces as a list of list of actions from the log.

In [120]:

traces = [[event['concept:name'] for event in trace] for trace in full_log]
print(f"{len(traces)} traces loaded from the log")
pp(traces[:1])


10357 traces loaded from the log
[['Declaration SUBMITTED by EMPLOYEE',
  'Declaration FINAL_APPROVED by SUPERVISOR',
  'Request Payment',
  'Payment Handled']]


## Apply a filter to the list traces

Get all the traces, where "(Employee) Increase employee satisfaction" is either strongly or weakly compliant.

In [121]:
trace_filter = TraceFilter(goal_model, traces, activity_mapping)

employee_satisfied_traces = (
    trace_filter
    .query()
    .where("(Employee) Increase employee satisfaction", ComplianceStatus.COMPLIANT)
    .traces()
)

print(f"{len(employee_satisfied_traces)} satisfy (strongly or weakly) the target '(Employee) Increase employee satisfaction'") 
print(f"This is {len(employee_satisfied_traces)*100/len(traces):.2f}% of all traces")
print(f"Only {(len(traces) - len(employee_satisfied_traces))*100/len(traces):.2f}% of traces do not satisfy the target '(Employee) Increase employee satisfaction'")

9909 satisfy (strongly or weakly) the target '(Employee) Increase employee satisfaction'
This is 95.67% of all traces
Only 4.33% of traces do not satisfy the target '(Employee) Increase employee satisfaction'


In [122]:
adequate_declaration_traces = (
    trace_filter
    .query()
    .where("(Employee) Increase employee satisfaction", ComplianceStatus.COMPLIANT)
    .where("(Admin) adequate declaration handling", ComplianceStatus.COMPLIANT)
    .traces()
)
print(f"Now {len(adequate_declaration_traces)*100/len(employee_satisfied_traces):.2f}% of those traces satisfy also the target '(Admin) adequate declaration handling'")
print(f"These are {len(adequate_declaration_traces)} traces")


Now 100.00% of those traces satisfy also the target '(Admin) adequate declaration handling'
These are 9909 traces


In [123]:
payment_but_no_declaration_submitted = (
    trace_filter
    .query()
    .contains("Payment Handled")
    .not_contains("Declaration SUBMITTED by EMPLOYEE")
    .sort_by_length()
    .traces()
)

pp(len(payment_but_no_declaration_submitted))
pp(payment_but_no_declaration_submitted)

1
[['Declaration SAVED by EMPLOYEE', 'Request Payment', 'Payment Handled']]


In [124]:
no_satisfaction_but_paid = (
    trace_filter
    .query()
    .where("(Employee) Increase employee satisfaction", ComplianceStatus.NON_COMPLIANT)
    .contains("Payment Handled")
    .sort_by_length()
    .traces()
)

pp(len(no_satisfaction_but_paid))
pp(no_satisfaction_but_paid)

3
[['Declaration SAVED by EMPLOYEE', 'Request Payment', 'Payment Handled'],
 ['Declaration SUBMITTED by EMPLOYEE',
  'Declaration REJECTED by SUPERVISOR',
  'Request Payment',
  'Payment Handled',
  'Declaration REJECTED by EMPLOYEE'],
 ['Declaration SUBMITTED by EMPLOYEE',
  'Declaration APPROVED by ADMINISTRATION',
  'Declaration REJECTED by SUPERVISOR',
  'Declaration SUBMITTED by EMPLOYEE',
  'Declaration APPROVED by ADMINISTRATION',
  'Declaration FINAL_APPROVED by SUPERVISOR',
  'Payment Handled']]


In [125]:

from Ui.interface import WhatIfInterfaceBuilder
from Ui.interface import WhatIfInterfaceBuilder

kogi_mapping = petri_net.convert_to_kogi_mapping(activity_mapping)
interface = WhatIfInterfaceBuilder(goal_model).create_interface()
display(interface)